# Chamfer Distance Comparison
This notebook constructs a sphere mesh, samples two point clouds from it using Conquer3D's GPU `TriangleMesh` data structure, and compares the Conquer3D GPU Chamfer Distance implementation against `point_cloud_utils`.

In [1]:
import torch
import conquer3d as c3d
import point_cloud_utils as pcu
import numpy as np
import time

# 1. Construct a sphere mesh
print("Constructing Sphere...")
verts, tris = c3d.creation.create_sphere(sectors=64, stacks=32, radius=1.0)
mesh = c3d.data_structure.TriangleMesh(verts.cuda(), tris.cuda())

print("Sampling Point Clouds using GPU TriangleMesh...")
N = 100000
torch.cuda.synchronize()
t0 = time.time()
pts1_cuda, *_ = mesh.sample_points(N, uniform=True)
pts2_cuda, *_ = mesh.sample_points(N, uniform=True)
torch.cuda.synchronize()
print(f"Sampled two point clouds of size {N} in {time.time()-t0:.4f}s")

Constructing Sphere...
Sampling Point Clouds using GPU TriangleMesh...
Sampled two point clouds of size 100000 in 0.0123s


In [2]:
# 2. Compute Chamfer Distance using Conquer3D
print("\n--- Conquer3D Chamfer Distance ---")
torch.cuda.synchronize()
t0 = time.time()
# Warmup
_ = c3d.ops.chamfer_distance(pts1_cuda[:100], pts2_cuda[:100], squared=False)
torch.cuda.synchronize()

t0 = time.time()
# We use squared=False to match point_cloud_utils' L2 norm
dist_c3d, idx_1_to_2, idx_2_to_1 = c3d.ops.chamfer_distance(pts1_cuda, pts2_cuda, squared=False, return_indices=True)
torch.cuda.synchronize()
t_c3d = time.time() - t0
dist_c3d_val = dist_c3d.item()
print(f"Conquer3D Distance (un-squared): {dist_c3d_val:.8f}")
print(f"Conquer3D Time:                  {t_c3d:.5f}s")


--- Conquer3D Chamfer Distance ---
Conquer3D Distance (un-squared): 0.01119104
Conquer3D Time:                  0.00728s


In [3]:
# 3. Compute Chamfer Distance using point_cloud_utils
print("\n--- Point Cloud Utils Chamfer Distance ---")
# PCU requires CPU numpy arrays
pts1_np = pts1_cuda.cpu().numpy()
pts2_np = pts2_cuda.cpu().numpy()

t0 = time.time()
dist_pcu = pcu.chamfer_distance(pts1_np, pts2_np)
t_pcu = time.time() - t0
print(f"PCU Distance:       {dist_pcu:.8f}")
print(f"PCU Time:           {t_pcu:.5f}s")

# 4. Compare Results
print("\n--- Comparison ---")
diff = abs(dist_c3d_val - dist_pcu)
print(f"Absolute Difference: {diff:.8e}")
print(f"Speedup:             {t_pcu / t_c3d:.2f}x")


--- Point Cloud Utils Chamfer Distance ---
PCU Distance:       0.01119104
PCU Time:           0.09412s

--- Comparison ---
Absolute Difference: 0.00000000e+00
Speedup:             12.93x
